# DNF cModel `griz` and `riz`: delivery QA

This report summarizes the scalar HATS products generated by the two DNF cModel runs. It checks catalog integrity, describes the delivered photo-z summaries, and quantifies two explicitly handled numerical or data-availability conditions.

This is **not** an accuracy validation against spectroscopic redshifts. In particular, the distributions below are distributions of photometric-redshift summaries, $N(z_{\rm phot})$, and must not be interpreted as calibrated estimates of the true population $N(z)$. No claims about bias, scatter, or outlier rate are made here.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import os
import subprocess
import sys

import dask
from dask import delayed
import h5py
from IPython.display import Markdown, display
import lsdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

VARIANTS = ("griz", "riz")
SUMMARY_COLUMNS = (
    "objectId", "coord_ra", "coord_dec",
    "dnf_z_mean", "dnf_z_median", "dnf_z_mode",
    "dnf_z_err68_low", "dnf_z_err68_high",
    "dnf_z_err95_low", "dnf_z_err95_high", "DNF_ZN",
)
SCIENCE_COLUMNS = SUMMARY_COLUMNS[3:]
GRID_COLUMNS = SCIENCE_COLUMNS[:-1]
HISTOGRAM_EDGES = np.linspace(0.0, 6.0, 121)
SAMPLE_DENOMINATOR = 4096

DP2_ROOT = Path(os.environ["PZ_DP2_ROOT"]).resolve()
RUN_ROOT = Path(os.environ["PZ_DP2_CUSTOM_ROOT"]).resolve()
INPUT_ROOT = Path(os.environ["PZ_DP2_INPUT_DATASET"]).resolve()
REPOSITORY = Path(os.environ["PZ_COMPUTE_DIR"]).resolve()
QA_OUTPUT = Path(os.environ.get("PZ_QA_OUTPUT_DIR", RUN_ROOT / "qa/delivery")).resolve()
QA_OUTPUT.mkdir(parents=True, exist_ok=True)

generated_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
commit = subprocess.check_output(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True
).strip()

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 6)

## Provenance and product scope

Both products contain object identity and coordinates, seven Rubin-style summaries derived from the DNF PDF, and `DNF_ZN`, the redshift of the selected DNF neighbor. `DNF_ZN` is not derived from the PDF. The final delivery intentionally contains no nested or companion PDF product.

In [ ]:
def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "not recorded"

validation_records = {}
for variant in VARIANTS:
    path = RUN_ROOT / variant / "stage3/production/VALIDATION.json"
    validation_records[variant] = json.loads(path.read_text(encoding="utf-8"))

provenance = pd.DataFrame(
    [
        ("Generated at (UTC)", generated_at),
        ("Repository commit", commit),
        ("Python", sys.version.split()[0]),
        ("NumPy", np.__version__),
        ("pandas", pd.__version__),
        ("Dask", package_version("dask")),
        ("LSDB", package_version("lsdb")),
        ("h5py", h5py.__version__),
        ("PDF grid", "601 points over the closed interval [0, 6]"),
        ("Deterministic comparison sample", f"approximately 1/{SAMPLE_DENOMINATOR} of objectId hashes"),
    ],
    columns=["item", "value"],
)
display(provenance.style.hide(axis="index"))

publication = pd.DataFrame(
    {
        variant: {
            "catalog": record["collection"],
            "rows": record["rows"],
            "columns": len(record["columns"]),
            "science dtype": record["science_dtype"],
            "PDF layout": record["pdf_layout"],
            "no-photometry policy": record["no_photometry_policy"],
        }
        for variant, record in validation_records.items()
    }
).T
display(publication)

## Catalog-wide integrity checks

The following counts scan every HATS row. A valid estimated row has eight finite science values. A row without usable selected-band photometry has all eight science values set to `NaN`. Partial null patterns are not allowed. Quantile summaries shown later use a deterministic sample; the integrity counts and histograms in this report are exact.

In [ ]:
def partition_metrics(frame: pd.DataFrame) -> dict:
    result = {
        "rows": len(frame),
        "coord_invalid": 0,
        "no_estimate_rows": 0,
        "partial_nonfinite_rows": 0,
        "interval_invalid_rows": 0,
        "grid_outside_values": 0,
        "all_summaries_at_zero_rows": 0,
        "all_summaries_at_six_rows": 0,
        "mode_at_zero_rows": 0,
        "mode_at_six_rows": 0,
        "dnf_zn_below_zero_rows": 0,
        "dnf_zn_above_six_rows": 0,
        "nonfinite": {name: 0 for name in SCIENCE_COLUMNS},
        "mean_histogram": np.zeros(len(HISTOGRAM_EDGES) - 1, dtype=np.int64),
        "zn_histogram": np.zeros(len(HISTOGRAM_EDGES) - 1, dtype=np.int64),
        "sample": frame.iloc[0:0][list(SUMMARY_COLUMNS)].copy(),
    }
    if not len(frame):
        return result

    ra = frame["coord_ra"].to_numpy(dtype=np.float64)
    dec = frame["coord_dec"].to_numpy(dtype=np.float64)
    result["coord_invalid"] = int(np.count_nonzero(
        ~np.isfinite(ra) | ~np.isfinite(dec) | (ra < 0) | (ra >= 360)
        | (dec < -90) | (dec > 90)
    ))

    values = {name: frame[name].to_numpy(dtype=np.float64) for name in SCIENCE_COLUMNS}
    science = np.column_stack([values[name] for name in SCIENCE_COLUMNS])
    grid = np.column_stack([values[name] for name in GRID_COLUMNS])
    no_estimate = np.isnan(science).all(axis=1)
    any_nonfinite = (~np.isfinite(science)).any(axis=1)
    result["no_estimate_rows"] = int(np.count_nonzero(no_estimate))
    result["partial_nonfinite_rows"] = int(np.count_nonzero(any_nonfinite & ~no_estimate))
    for name, array in values.items():
        result["nonfinite"][name] = int(np.count_nonzero(~np.isfinite(array)))

    ordered = (
        (values["dnf_z_err95_low"] <= values["dnf_z_err68_low"])
        & (values["dnf_z_err68_low"] <= values["dnf_z_median"])
        & (values["dnf_z_median"] <= values["dnf_z_err68_high"])
        & (values["dnf_z_err68_high"] <= values["dnf_z_err95_high"])
    )
    result["interval_invalid_rows"] = int(np.count_nonzero(~ordered & ~no_estimate))
    result["grid_outside_values"] = int(np.count_nonzero((grid < 0) | (grid > 6)))
    estimated = ~no_estimate
    result["all_summaries_at_zero_rows"] = int(np.count_nonzero(estimated & np.all(grid == 0, axis=1)))
    result["all_summaries_at_six_rows"] = int(np.count_nonzero(estimated & np.all(grid == 6, axis=1)))
    result["mode_at_zero_rows"] = int(np.count_nonzero(values["dnf_z_mode"] == 0))
    result["mode_at_six_rows"] = int(np.count_nonzero(values["dnf_z_mode"] == 6))

    zn = values["DNF_ZN"]
    result["dnf_zn_below_zero_rows"] = int(np.count_nonzero(zn < 0))
    result["dnf_zn_above_six_rows"] = int(np.count_nonzero(zn > 6))
    mean = values["dnf_z_mean"]
    result["mean_histogram"] = np.histogram(mean[np.isfinite(mean)], bins=HISTOGRAM_EDGES)[0]
    result["zn_histogram"] = np.histogram(zn[np.isfinite(zn)], bins=HISTOGRAM_EDGES)[0]

    hashes = pd.util.hash_pandas_object(frame["objectId"], index=False).to_numpy(dtype=np.uint64)
    selected = hashes % SAMPLE_DENOMINATOR == 0
    result["sample"] = frame.loc[selected, list(SUMMARY_COLUMNS)].copy()
    return result


def catalog_metrics(variant: str) -> tuple[dict, pd.DataFrame]:
    path = RUN_ROOT / variant / "stage3/output" / f"pz-dnf-cmodel-{variant}-scalar-hats"
    catalog = lsdb.open_catalog(path)
    schema_exact = list(catalog.columns) == list(SUMMARY_COLUMNS)
    if not schema_exact:
        raise RuntimeError(f"{variant}: unexpected schema")
    dtypes = catalog[list(SCIENCE_COLUMNS)].dtypes
    science_float64 = all(
        np.dtype(getattr(dtypes[name], "numpy_dtype", dtypes[name])) == np.dtype(np.float64)
        for name in SCIENCE_COLUMNS
    )
    nested_columns_count = len(catalog.nested_columns)
    margin_arcsec = (
        None if catalog.margin is None
        else float(catalog.margin.hc_structure.catalog_info.margin_threshold)
    )
    frame = catalog[list(SUMMARY_COLUMNS)].to_dask_dataframe()
    tasks = [delayed(partition_metrics)(part) for part in frame.to_delayed()]
    with dask.config.set(scheduler="threads", num_workers=2):
        parts = list(dask.compute(*tasks))

    result = {key: 0 for key in (
        "rows", "coord_invalid", "no_estimate_rows",
        "partial_nonfinite_rows", "interval_invalid_rows",
        "grid_outside_values", "all_summaries_at_zero_rows",
        "all_summaries_at_six_rows", "mode_at_zero_rows",
        "mode_at_six_rows", "dnf_zn_below_zero_rows",
        "dnf_zn_above_six_rows",
    )}
    result["metadata_rows"] = len(catalog)
    result["nonfinite"] = {name: 0 for name in SCIENCE_COLUMNS}
    result["mean_histogram"] = np.zeros(len(HISTOGRAM_EDGES) - 1, dtype=np.int64)
    result["zn_histogram"] = np.zeros(len(HISTOGRAM_EDGES) - 1, dtype=np.int64)
    samples = []
    for part in parts:
        for key in result:
            if key in ("metadata_rows", "nonfinite", "mean_histogram", "zn_histogram"):
                continue
            result[key] += part[key]
        for name in SCIENCE_COLUMNS:
            result["nonfinite"][name] += part["nonfinite"][name]
        result["mean_histogram"] += part["mean_histogram"]
        result["zn_histogram"] += part["zn_histogram"]
        samples.append(part["sample"])
    sample = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=SUMMARY_COLUMNS)
    result["schema_exact"] = schema_exact
    result["science_float64"] = science_float64
    result["nested_columns_count"] = nested_columns_count
    result["margin_arcsec"] = margin_arcsec
    result["sample_rows"] = len(sample)
    result["sample_duplicate_object_ids"] = int(sample["objectId"].duplicated().sum())
    return result, sample

hats_metrics = {}
samples = {}
for variant in VARIANTS:
    hats_metrics[variant], samples[variant] = catalog_metrics(variant)

In [ ]:
integrity_rows = []
for variant in VARIANTS:
    metrics = hats_metrics[variant]
    expected_no_estimate = int(validation_records[variant]["no_estimate_rows"])
    integrity_rows.extend([
        (variant, "exact ordered schema", metrics["schema_exact"], len(SUMMARY_COLUMNS)),
        (variant, "all science columns are float64", metrics["science_float64"], len(SCIENCE_COLUMNS)),
        (variant, "nested columns absent", metrics["nested_columns_count"] == 0, metrics["nested_columns_count"]),
        (variant, "default margin matches publication record", np.isclose(metrics["margin_arcsec"], float(validation_records[variant]["default_margin_arcsec"])), metrics["margin_arcsec"]),
        (variant, "scanned rows equal HATS metadata", metrics["rows"] == metrics["metadata_rows"], metrics["rows"]),
        (variant, "rows equal publication record", metrics["rows"] == int(validation_records[variant]["rows"]), metrics["rows"]),
        (variant, "coordinates outside contract", metrics["coord_invalid"] == 0, metrics["coord_invalid"]),
        (variant, "partial non-finite science rows", metrics["partial_nonfinite_rows"] == 0, metrics["partial_nonfinite_rows"]),
        (variant, "unordered interval rows", metrics["interval_invalid_rows"] == 0, metrics["interval_invalid_rows"]),
        (variant, "PDF-derived values outside [0, 6]", metrics["grid_outside_values"] == 0, metrics["grid_outside_values"]),
        (variant, "no-estimate rows match publication record", metrics["no_estimate_rows"] == expected_no_estimate, metrics["no_estimate_rows"]),
        (variant, "duplicate objectId values in deterministic sample", metrics["sample_duplicate_object_ids"] == 0, metrics["sample_duplicate_object_ids"]),
    ])
    expected_nulls = metrics["no_estimate_rows"]
    for name, count in metrics["nonfinite"].items():
        integrity_rows.append((variant, f"{name} non-finite count matches policy", count == expected_nulls, count))

integrity = pd.DataFrame(integrity_rows, columns=["variant", "check", "passed", "count"])
integrity["status"] = np.where(integrity["passed"], "PASS", "FAIL")
display(integrity[["variant", "check", "status", "count"]].style.hide(axis="index"))
if not integrity["passed"].all():
    raise RuntimeError("one or more delivery QA integrity checks failed")

The duplicate-ID check above applies to the deterministic comparison sample, not to an independent full-catalog uniqueness scan. The production workflow checked Stage 2 row-identity fingerprints and Stage 2/HATS row counts; the sample check here is an additional inexpensive diagnostic and is not presented as proof of full-catalog uniqueness.

## Distributions of delivered photo-z summaries

The histograms below use every finite catalog value and common bins. `dnf_z_mean` is the mean of the discretized PDF. `DNF_ZN` is the redshift of the selected DNF neighbor and is not constrained by construction to be a PDF-grid summary. Counts outside the plotted `[0, 6]` range are reported separately.

In [ ]:
centers = 0.5 * (HISTOGRAM_EDGES[:-1] + HISTOGRAM_EDGES[1:])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for variant, color in zip(VARIANTS, ("tab:blue", "tab:orange")):
    for axis, key, label in (
        (axes[0], "mean_histogram", r"$z_{\rm mean}$"),
        (axes[1], "zn_histogram", r"DNF $Z_N$"),
    ):
        counts = hats_metrics[variant][key]
        finite_rows = hats_metrics[variant]["rows"] - hats_metrics[variant]["no_estimate_rows"]
        density = counts / finite_rows if finite_rows else counts
        axis.step(centers, density, where="mid", label=variant, color=color)
        axis.set_title(f"Distribution of {label}")
        axis.set_xlabel("photometric-redshift summary")
        axis.set_ylabel("fraction of finite catalog per 0.05 bin")
        axis.set_xlim(0, 6)
        axis.legend()
fig.tight_layout()
plt.show()

quantile_rows = []
quantiles = [0.0, 0.01, 0.16, 0.50, 0.84, 0.99, 1.0]
for variant in VARIANTS:
    for name in ("dnf_z_mean", "DNF_ZN"):
        values = samples[variant][name].to_numpy(dtype=np.float64)
        values = values[np.isfinite(values)]
        row = {"variant": variant, "quantity": name, "sample rows": len(values)}
        row.update({label: value for label, value in zip(("min", "p1", "p16", "median", "p84", "p99", "max"), np.quantile(values, quantiles))})
        quantile_rows.append(row)
display(Markdown("**Descriptive quantiles from the deterministic hash sample**"))
display(pd.DataFrame(quantile_rows).style.hide(axis="index").format(precision=4))

outside_zn = pd.DataFrame(
    {variant: {
        "DNF_ZN < 0": hats_metrics[variant]["dnf_zn_below_zero_rows"],
        "DNF_ZN > 6": hats_metrics[variant]["dnf_zn_above_six_rows"],
    } for variant in VARIANTS}
).T
display(outside_zn)

## Internal comparison of `griz` and `riz`

The comparison uses the same deterministic object-ID hash selection in both catalogs and then matches objects by `objectId`. It is descriptive only: agreement between the two band selections does not establish accuracy.

In [ ]:
paired = samples["griz"].merge(
    samples["riz"], on="objectId", how="inner", suffixes=("_griz", "_riz"), validate="one_to_one"
)
griz_mean = paired["dnf_z_mean_griz"].to_numpy(dtype=np.float64)
riz_mean = paired["dnf_z_mean_riz"].to_numpy(dtype=np.float64)
both = np.isfinite(griz_mean) & np.isfinite(riz_mean)
griz_only = np.isfinite(griz_mean) & ~np.isfinite(riz_mean)
riz_only = ~np.isfinite(griz_mean) & np.isfinite(riz_mean)
neither = ~np.isfinite(griz_mean) & ~np.isfinite(riz_mean)

sample_overlap = pd.DataFrame({
    "count": [len(paired), int(both.sum()), int(griz_only.sum()), int(riz_only.sum()), int(neither.sum())]
}, index=["matched sample rows", "finite in both", "finite only in griz", "finite only in riz", "NaN in both"])
display(sample_overlap)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hexbin = axes[0].hexbin(griz_mean[both], riz_mean[both], gridsize=80, bins="log", mincnt=1, extent=(0, 6, 0, 6))
axes[0].plot([0, 6], [0, 6], color="black", linewidth=1, linestyle="--")
axes[0].set(xlabel=r"$z_{\rm mean}$ (`griz`)", ylabel=r"$z_{\rm mean}$ (`riz`)", title="Deterministic matched sample")
fig.colorbar(hexbin, ax=axes[0], label="log-scaled count")
delta = griz_mean[both] - riz_mean[both]
axes[1].hist(delta, bins=100, histtype="step", color="tab:purple")
axes[1].axvline(0, color="black", linewidth=1, linestyle="--")
axes[1].set(xlabel=r"$z_{\rm mean}(griz)-z_{\rm mean}(riz)$", ylabel="sample objects", title="Difference between variants")
fig.tight_layout()
plt.show()

## Persisted Stage 2 quality counters

These counters were accumulated during estimation and stored in each Stage 2 HDF5 file. Reading them does not reconstruct or reinterpret the estimator. `legacy_gaussian_all_zero_rows` counts rows for which direct evaluation of the Gaussian would underflow to zero at every grid point. The delivered implementation evaluates the same fitted Gaussian in log-space before normalization. The counter therefore measures exposure to the legacy numerical mechanism; it does not mean that the delivered PDF is invalid.

In [ ]:
PDF_COUNTERS = (
    "pdf_rows", "nonfinite_dnf_z_rows", "nonfinite_photozerr_rows",
    "nonpositive_photozerr_rows", "dnf_z_below_grid_rows",
    "dnf_z_above_grid_rows", "no_estimate_rows",
    "legacy_gaussian_all_zero_rows", "pdf_nonfinite_rows",
    "pdf_nonpositive_integral_rows", "pdf_bad_normalization_rows",
    "pdf_mode_at_lower_grid_rows", "pdf_mode_at_upper_grid_rows",
)
NUMERICAL_FAILURE_COUNTERS = (
    "nonfinite_dnf_z_rows", "nonfinite_photozerr_rows",
    "nonpositive_photozerr_rows", "pdf_nonfinite_rows",
    "pdf_nonpositive_integral_rows", "pdf_bad_normalization_rows",
)

manifest = DP2_ROOT / "common/manifests/input-parquet-files.txt"
input_paths = [Path(line) for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()]

def aggregate_stage2(variant: str) -> dict:
    counters = {name: 0 for name in PDF_COUNTERS}
    input_counters = {}
    legacy_affected_shards = 0
    legacy_max_per_shard = 0
    selected_bands = None
    policy = None
    for input_path in input_paths:
        relative = input_path.relative_to(INPUT_ROOT)
        output = RUN_ROOT / variant / "stage2/output" / relative.with_suffix(".hdf5")
        with h5py.File(output, "r") as handle:
            pdf = handle["pz_compute/dnf_pdf_quality"].attrs
            quality = handle["pz_compute/dnf_input_quality"].attrs
            for name in PDF_COUNTERS:
                counters[name] += int(pdf[name])
            legacy = int(pdf["legacy_gaussian_all_zero_rows"])
            legacy_affected_shards += int(legacy > 0)
            legacy_max_per_shard = max(legacy_max_per_shard, legacy)
            for name in quality:
                if name.startswith(("missing_", "unusable_")) or name in (
                    "rows", "complete_selected_band_nondetection_rows",
                    "no_usable_selected_band_rows",
                ):
                    input_counters[name] = input_counters.get(name, 0) + int(quality[name])
            shard_bands = str(quality["selected_bands"])
            shard_policy = str(quality["no_photometry_policy"])
            selected_bands = shard_bands if selected_bands is None else selected_bands
            policy = shard_policy if policy is None else policy
            if selected_bands != shard_bands or policy != shard_policy:
                raise RuntimeError(f"{variant}: inconsistent Stage 2 quality metadata")
    return {
        "shards": len(input_paths),
        "selected_bands": selected_bands,
        "no_photometry_policy": policy,
        "pdf_counters": counters,
        "input_counters": input_counters,
        "legacy_underflow_affected_shards": legacy_affected_shards,
        "legacy_underflow_max_per_shard": legacy_max_per_shard,
    }

stage2_metrics = {variant: aggregate_stage2(variant) for variant in VARIANTS}

In [ ]:
underflow_rows = []
for variant in VARIANTS:
    metrics = stage2_metrics[variant]
    counters = metrics["pdf_counters"]
    estimated = counters["pdf_rows"] - counters["no_estimate_rows"]
    underflow_rows.append({
        "variant": variant,
        "PDF rows": counters["pdf_rows"],
        "estimated rows": estimated,
        "legacy all-zero rows": counters["legacy_gaussian_all_zero_rows"],
        "legacy fraction of estimated rows": counters["legacy_gaussian_all_zero_rows"] / estimated if estimated else np.nan,
        "affected shards": metrics["legacy_underflow_affected_shards"],
        "maximum in one shard": metrics["legacy_underflow_max_per_shard"],
    })
underflow = pd.DataFrame(underflow_rows)
display(underflow.style.hide(axis="index").format({"legacy fraction of estimated rows": "{:.6%}"}))

numerical_rows = []
for variant in VARIANTS:
    counters = stage2_metrics[variant]["pdf_counters"]
    numerical_rows.append({"variant": variant, **{name: counters[name] for name in NUMERICAL_FAILURE_COUNTERS}})
numerical = pd.DataFrame(numerical_rows)
display(Markdown("**Persisted numerical-failure counters for delivered estimates**"))
display(numerical.style.hide(axis="index"))
if numerical[list(NUMERICAL_FAILURE_COUNTERS)].to_numpy().any():
    raise RuntimeError("persisted DNF numerical-failure counters are nonzero")

## Rows without usable selected-band photometry

A row enters the no-estimate policy only when none of its selected bands has both a usable magnitude and a usable magnitude error before non-detection replacement. Such rows remain in the catalog to preserve membership and identity, but all eight delivered photo-z fields, including `DNF_ZN`, are `NaN`. This avoids presenting an estimate based only on an identical imputed limiting-magnitude vector as object-specific information.

In [ ]:
missing_rows = []
for variant in VARIANTS:
    metrics = stage2_metrics[variant]
    inputs = metrics["input_counters"]
    total = inputs["rows"]
    for band in metrics["selected_bands"]:
        missing_rows.append({
            "variant": variant, "band": band, "total rows": total,
            "missing magnitude rows": inputs[f"missing_{band}_rows"],
            "unusable magnitude/error rows": inputs[f"unusable_{band}_rows"],
        })
display(pd.DataFrame(missing_rows).style.hide(axis="index"))

policy_rows = []
for variant in VARIANTS:
    inputs = stage2_metrics[variant]["input_counters"]
    total = inputs["rows"]
    no_estimate = inputs["no_usable_selected_band_rows"]
    policy_rows.append({
        "variant": variant,
        "catalog rows": total,
        "all selected magnitudes missing": inputs["complete_selected_band_nondetection_rows"],
        "no usable selected band": no_estimate,
        "no-estimate fraction": no_estimate / total if total else np.nan,
        "HATS all-NaN science rows": hats_metrics[variant]["no_estimate_rows"],
        "HATS partial-NaN rows": hats_metrics[variant]["partial_nonfinite_rows"],
    })
policy = pd.DataFrame(policy_rows)
display(policy.style.hide(axis="index").format({"no-estimate fraction": "{:.6%}"}))
for row in policy_rows:
    if row["no usable selected band"] != row["HATS all-NaN science rows"] or row["HATS partial-NaN rows"]:
        raise RuntimeError(f"{row['variant']}: HATS null pattern differs from the input-derived policy")

## Boundary-saturated PDF summaries

When all seven PDF-derived summaries equal a grid boundary, the discretized PDF is concentrated at the edge of its supported interval. This should be interpreted as a boundary-saturated estimate, not as a physical measurement with zero uncertainty. The weighted mean is bounded to the closed PDF support only to remove floating-point excursions such as the next representable value above `6.0`; this does not alter the fitted DNF point estimates or the PDF.

In [ ]:
boundary_rows = []
for variant in VARIANTS:
    hats = hats_metrics[variant]
    pdf = stage2_metrics[variant]["pdf_counters"]
    boundary_rows.append({
        "variant": variant,
        "all seven summaries at 0": hats["all_summaries_at_zero_rows"],
        "all seven summaries at 6": hats["all_summaries_at_six_rows"],
        "HATS mode at 0": hats["mode_at_zero_rows"],
        "HATS mode at 6": hats["mode_at_six_rows"],
        "fitted DNF_Z below grid": pdf["dnf_z_below_grid_rows"],
        "fitted DNF_Z above grid": pdf["dnf_z_above_grid_rows"],
        "Stage 2 PDF mode at 0": pdf["pdf_mode_at_lower_grid_rows"],
        "Stage 2 PDF mode at 6": pdf["pdf_mode_at_upper_grid_rows"],
    })
display(pd.DataFrame(boundary_rows).style.hide(axis="index"))

## Machine-readable QA record and conclusion

The accompanying JSON preserves the aggregate values displayed above. Individual sampled object rows are intentionally not included in that record.

In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

public_hats_metrics = {
    variant: {key: value for key, value in metrics.items() if key != "sample"}
    for variant, metrics in hats_metrics.items()
}
qa_record = {
    "schema_version": 1,
    "generated_at_utc": generated_at,
    "repository_commit": commit,
    "scope": "descriptive delivery QA; no spectroscopic truth validation",
    "sample": {"method": "pandas objectId hash modulo", "denominator": SAMPLE_DENOMINATOR},
    "stage3_validation_records": validation_records,
    "hats_metrics": public_hats_metrics,
    "stage2_metrics": stage2_metrics,
}
summary_path = QA_OUTPUT / "qa_summary.json"
summary_path.write_text(json.dumps(json_ready(qa_record), indent=2, sort_keys=True) + "\n", encoding="utf-8")

lines = ["All catalog-wide integrity checks passed for both variants."]
for variant in VARIANTS:
    pdf = stage2_metrics[variant]["pdf_counters"]
    inputs = stage2_metrics[variant]["input_counters"]
    lines.append(
        f"- **{variant}**: {hats_metrics[variant]['rows']:,} catalog rows; "
        f"{inputs['no_usable_selected_band_rows']:,} rows without a usable selected band; "
        f"{pdf['legacy_gaussian_all_zero_rows']:,} estimated rows exposed to the legacy all-zero Gaussian underflow mechanism."
    )
lines.append("The report establishes structural consistency and records known numerical/data-availability populations. It does not establish redshift accuracy or calibration.")
display(Markdown("\n".join(lines)))
print("Machine-readable QA record: qa_summary.json")